In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -U --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

In [3]:
import os
import copy
import random
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models


# =========================================================
# CONFIG
# =========================================================
class CFG:
    # Change this path to your dataset root
    DATA_ROOT = "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"

    TRAIN_DIR = os.path.join(DATA_ROOT, "Training")
    TEST_DIR  = os.path.join(DATA_ROOT, "Testing")

    IMG_SIZE = 224
    BATCH_SIZE = 16
    EPOCHS = 3
    LR = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    SEED = 42
    USE_WEIGHTED_SAMPLER = True
    EARLY_STOPPING_PATIENCE = 5

    MODEL_PATH = "/kaggle/working/best_brain_tumor_efficientnet_b3.pth"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# =========================================================
# LABELS
# =========================================================
LABEL_TO_INDEX = {
    "glioma": 0,
    "meningioma": 1,
    "notumor": 2,
    "pituitary": 3
}
INDEX_TO_LABEL = {v: k for k, v in LABEL_TO_INDEX.items()}

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


# =========================================================
# REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# DATA COLLECTION
# =========================================================
def collect_samples(split_dir):
    split_dir = Path(split_dir)
    samples = []

    print(f"\nScanning: {split_dir}")
    print(f"Exists: {split_dir.exists()}")

    for class_name in LABEL_TO_INDEX.keys():
        class_dir = split_dir / class_name
        if not class_dir.exists():
            print(f"Warning: missing folder -> {class_dir}")
            continue

        image_files = sorted([p for p in class_dir.glob("*") if p.suffix.lower() in VALID_EXTENSIONS])
        print(f"{class_name}: {len(image_files)} images")

        label_idx = LABEL_TO_INDEX[class_name]
        for img_path in image_files:
            samples.append((str(img_path), label_idx))

    return samples


# =========================================================
# DATASET
# =========================================================
class BrainTumorDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


# =========================================================
# TRANSFORMS
# =========================================================
def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    valid_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    return train_transform, valid_transform


# =========================================================
# DATALOADERS
# =========================================================
def make_loaders(train_samples, test_samples):
    train_tfms, valid_tfms = get_transforms(CFG.IMG_SIZE)

    train_ds = BrainTumorDataset(train_samples, transform=train_tfms)
    test_ds = BrainTumorDataset(test_samples, transform=valid_tfms)

    if CFG.USE_WEIGHTED_SAMPLER:
        train_labels = [label for _, label in train_samples]
        class_count = np.bincount(train_labels, minlength=len(LABEL_TO_INDEX))
        class_weights = 1.0 / np.maximum(class_count, 1)
        sample_weights = [class_weights[label] for label in train_labels]

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights),
            replacement=True
        )

        train_loader = DataLoader(
            train_ds,
            batch_size=CFG.BATCH_SIZE,
            sampler=sampler,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=CFG.BATCH_SIZE,
            shuffle=True,
            num_workers=CFG.NUM_WORKERS,
            pin_memory=True
        )

    test_loader = DataLoader(
        test_ds,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=True
    )

    return train_loader, test_loader


# =========================================================
# MODEL
# =========================================================
def build_model(num_classes=4):
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model


# =========================================================
# METRICS
# =========================================================
def compute_metrics(labels_all, preds_all):
    return {
        "acc": accuracy_score(labels_all, preds_all),
        "f1": f1_score(labels_all, preds_all, average="macro"),
        "precision": precision_score(labels_all, preds_all, average="macro", zero_division=0),
        "recall": recall_score(labels_all, preds_all, average="macro", zero_division=0),
    }


# =========================================================
# TRAIN / VALIDATE
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device, epoch):
    model.train()
    running_loss = 0.0
    preds_all = []
    labels_all = []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Train]", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

        metrics = compute_metrics(labels_all, preds_all)
        pbar.set_postfix(
            loss=f"{running_loss / len(labels_all):.4f}",
            acc=f"{metrics['acc']:.4f}",
            f1=f"{metrics['f1']:.4f}"
        )

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(labels_all, preds_all)
    return epoch_loss, metrics


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, epoch):
    model.eval()
    running_loss = 0.0
    preds_all = []
    labels_all = []

    pbar = tqdm(loader, desc=f"Epoch {epoch} [Test]", leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)
        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(labels.detach().cpu().numpy())

        metrics = compute_metrics(labels_all, preds_all)
        pbar.set_postfix(
            loss=f"{running_loss / len(labels_all):.4f}",
            acc=f"{metrics['acc']:.4f}",
            f1=f"{metrics['f1']:.4f}"
        )

    epoch_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(labels_all, preds_all)
    return epoch_loss, metrics, labels_all, preds_all


# =========================================================
# FINAL EVALUATION
# =========================================================
def evaluate_predictions(labels_all, preds_all):
    print("\nFinal Test Accuracy:", accuracy_score(labels_all, preds_all))
    print("Final Test Macro F1:", f1_score(labels_all, preds_all, average="macro"))

    target_names = [INDEX_TO_LABEL[i] for i in range(len(INDEX_TO_LABEL))]
    print("\nClassification Report:")
    print(classification_report(labels_all, preds_all, target_names=target_names, zero_division=0))

    print("Confusion Matrix:")
    print(confusion_matrix(labels_all, preds_all))


# =========================================================
# SINGLE IMAGE PREDICTION
# =========================================================
@torch.no_grad()
def predict_image(model, image_path, device):
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    image = Image.open(image_path).convert("RGB")
    image_tensor = transform(image).unsqueeze(0).to(device)

    outputs = model(image_tensor)
    probs = torch.softmax(outputs, dim=1)[0]
    pred_idx = torch.argmax(probs).item()

    pred_label = INDEX_TO_LABEL[pred_idx]
    confidence = probs[pred_idx].item()
    all_probs = {INDEX_TO_LABEL[i]: float(p) for i, p in enumerate(probs.cpu().numpy())}

    print(f"Prediction: {pred_label}")
    print(f"Confidence: {confidence:.4f}")
    print(f"All probabilities: {all_probs}")

    return pred_label, confidence, all_probs


# =========================================================
# MAIN
# =========================================================
def main():
    set_seed(CFG.SEED)
    print("Using device:", CFG.DEVICE)

    train_samples = collect_samples(CFG.TRAIN_DIR)
    test_samples = collect_samples(CFG.TEST_DIR)

    print(f"\nTrain samples: {len(train_samples)}")
    print(f"Test samples:  {len(test_samples)}")

    if len(train_samples) == 0 or len(test_samples) == 0:
        raise ValueError("Train or test samples not found. Check folder structure.")

    train_loader, test_loader = make_loaders(train_samples, test_samples)

    model = build_model(num_classes=4).to(CFG.DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_f1 = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    early_stop_counter = 0

    epoch_bar = tqdm(range(1, CFG.EPOCHS + 1), desc="Training Progress")

    for epoch in epoch_bar:
        train_loss, train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, CFG.DEVICE, epoch
        )

        test_loss, test_metrics, test_labels, test_preds = validate_one_epoch(
            model, test_loader, criterion, CFG.DEVICE, epoch
        )

        scheduler.step(test_metrics["f1"])

        print(
            f"\nEpoch [{epoch}/{CFG.EPOCHS}] "
            f"| Train Loss: {train_loss:.4f} | Train Acc: {train_metrics['acc']:.4f} | Train F1: {train_metrics['f1']:.4f} "
            f"| Test Loss: {test_loss:.4f} | Test Acc: {test_metrics['acc']:.4f} | Test F1: {test_metrics['f1']:.4f} "
            f"| Test Precision: {test_metrics['precision']:.4f} | Test Recall: {test_metrics['recall']:.4f}"
        )

        epoch_bar.set_postfix(best_test_f1=f"{best_f1:.4f}", current_test_f1=f"{test_metrics['f1']:.4f}")

        if test_metrics["f1"] > best_f1:
            best_f1 = test_metrics["f1"]
            best_weights = copy.deepcopy(model.state_dict())
            torch.save(best_weights, CFG.MODEL_PATH)
            print(f"Best model saved to {CFG.MODEL_PATH}")
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            print(f"No improvement. Early stop counter: {early_stop_counter}/{CFG.EARLY_STOPPING_PATIENCE}")

        if early_stop_counter >= CFG.EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            break

    print("\nLoading best model...")
    model.load_state_dict(best_weights)

    print("\nEvaluating best model on test set...")
    _, _, test_labels, test_preds = validate_one_epoch(
        model, test_loader, criterion, CFG.DEVICE, epoch=999
    )
    evaluate_predictions(test_labels, test_preds)

    # Example single image prediction after training
    # sample_image = "/kaggle/input/brain-tumor-mri-dataset/Brain Tumor MRI Dataset/Testing/glioma/some_image.jpg"
    # predict_image(model, sample_image, CFG.DEVICE)


if __name__ == "__main__":
    main()

Using device: cuda

Scanning: /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training
Exists: True
glioma: 1400 images
meningioma: 1400 images
notumor: 1400 images
pituitary: 1400 images

Scanning: /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing
Exists: True
glioma: 400 images
meningioma: 400 images
notumor: 400 images
pituitary: 400 images

Train samples: 5600
Test samples:  1600


Training Progress:   0%|          | 0/3 [00:00<?, ?it/s]

Epoch 1 [Train]:   0%|          | 0/350 [00:00<?, ?it/s]

Epoch 1 [Test]:   0%|          | 0/100 [00:00<?, ?it/s]


Epoch [1/3] | Train Loss: 0.4268 | Train Acc: 0.8609 | Train F1: 0.8599 | Test Loss: 0.2726 | Test Acc: 0.9194 | Test F1: 0.9175 | Test Precision: 0.9230 | Test Recall: 0.9194
Best model saved to /kaggle/working/best_brain_tumor_efficientnet_b3.pth


Epoch 2 [Train]:   0%|          | 0/350 [00:00<?, ?it/s]

... (4 outputs left out)


Epoch [3/3] | Train Loss: 0.0854 | Train Acc: 0.9727 | Train F1: 0.9726 | Test Loss: 0.2478 | Test Acc: 0.9450 | Test F1: 0.9438 | Test Precision: 0.9487 | Test Recall: 0.9450
Best model saved to /kaggle/working/best_brain_tumor_efficientnet_b3.pth

Loading best model...

Evaluating best model on test set...


Epoch 999 [Test]:   0%|          | 0/100 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d53404c7ba0><function _MultiProcessingDataLoaderIter.__del__ at 0x7d53404c7ba0>

Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
        self._shutdown_workers()self._shutdown_workers()

  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
        if w.is_alive():if w.is_alive():
 
      Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d53404c7ba0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 


Final Test Accuracy: 0.945
Final Test Macro F1: 0.9438148604111738

Classification Report:
              precision    recall  f1-score   support

      glioma       1.00      0.81      0.90       400
  meningioma       0.89      0.98      0.93       400
     notumor       0.94      1.00      0.97       400
   pituitary       0.97      0.99      0.98       400

    accuracy                           0.94      1600
   macro avg       0.95      0.95      0.94      1600
weighted avg       0.95      0.94      0.94      1600

Confusion Matrix:
[[326  47  22   5]
 [  0 391   2   7]
 [  1   0 399   0]
 [  0   2   2 396]]
